# 05 — Graph control, mapping, and serialization

**Objectives.** Compare SX/MX graphs, eliminate repeated Python loops, inspect functions, plot computational graphs, and persist graphs. Graph design often matters more than individual arithmetic operations.


In [4]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Build, roll out, and serialize a discrete state-transition graph

The cell creates an MX state-transition `Function`. `mapaccum` copies that
function across eight stages and connects each output state to the next
stage's input, producing a symbolic recurrence without a Python loop.
`serialize()` converts the graph to CasADi's textual representation, and
`Function.deserialize()` reconstructs it. Serialization should be used
only across compatible CasADi versions.

The one-step model is

$$
x_{k+1}=
\begin{bmatrix}1&0.1\\0&1\end{bmatrix}x_k
+\begin{bmatrix}0\\0.1\end{bmatrix}u_k.
$$

`mapaccum` represents the recurrence

$$
x_{k+1}=F(x_k,u_k),\qquad k=0,\ldots,7.
$$

The infinity norm verifies that the restored and original functions return
identical values for a representative input.

The two-panel time-history figure plots both components of the state $x_k$ and the held manipulated input $u_k$, making the recurrence visible rather than reporting only its final matrix shape.


In [ ]:
import casadi as ca
import matplotlib.pyplot as plt
import numpy as np

x = ca.MX.sym("x", 2)
u = ca.MX.sym("u")
x_next = ca.vertcat(
    x[0] + 0.1 * x[1],
    x[1] + 0.1 * u,
)
step = ca.Function(
    "step",
    [x, u],
    [x_next],
    ["x", "u"],
    ["x_next"],
)

# mapaccum feeds each x_next output into the following stage's x input.
horizon_steps = 8
step_size = 0.1
rollout = step.mapaccum("rollout", horizon_steps)
u_sequence = ca.DM.ones(1, horizon_steps)
x_prediction = rollout([0, 0], u_sequence)
assert x_prediction.shape == (2, horizon_steps)
print(step.info(), "nodes:", step.n_nodes())

# Include x_0 so the plotted state history spans the complete horizon.
x_history = np.column_stack([np.zeros(2), np.asarray(x_prediction.full())])
x_time = step_size * np.arange(horizon_steps + 1)
u_time = step_size * np.arange(horizon_steps)

figure, axes = plt.subplots(2, 1, sharex=True, figsize=(7, 5))
axes[0].plot(x_time, x_history[0], marker="o", label="x[0]")
axes[0].plot(x_time, x_history[1], marker="o", label="x[1]")
axes[0].set_ylabel("state x")
axes[0].legend()
axes[0].grid(True)
axes[1].step(
    u_time,
    np.asarray(u_sequence.full()).ravel(),
    where="post",
    label="u",
)
axes[1].set(xlabel="time", ylabel="input u")
axes[1].grid(True)
figure.suptitle("Mapped state-transition rollout")
figure.tight_layout()
plt.show()

# serialize records the Function graph; deserialize reconstructs it.
serialized_step = step.serialize()
restored_step = ca.Function.deserialize(serialized_step)
difference = restored_step([0, 0], 1) - step([0, 0], 1)
assert float(ca.norm_inf(difference)) == 0.0

### Render the computational graph with CasADi's native graph utility

`dotdraw` operates on an SX or MX expression rather than directly on a
`Function`. The cell therefore calls the step function symbolically, then
sends the resulting expression to `casadi.tools.graph.dotdraw`. The `LR`
layout directs Graphviz to arrange dependencies from left to right.

The displayed directed acyclic graph represents the composition

$$
(x_k,u_k)\longrightarrow
\begin{bmatrix}
x_{0,k}+0.1x_{1,k}\\
x_{1,k}+0.1u_k
\end{bmatrix}
\longrightarrow x_{k+1}.
$$

Nodes correspond to inputs, constants, indexing operations, arithmetic
operations, concatenation, and the final matrix-valued output. The arrows
show data dependencies, not the passage of simulation time.


In [ ]:
from casadi.tools.graph import dotdraw

# dotdraw traverses an SX or MX expression, rather than a Function object.
x_next_expression = step(x, u)

# Graphviz arranges dependencies from x and u to x_next from left to right.
dotdraw(x_next_expression, direction="LR")

`dotdraw` accepts an SX or MX expression, so the function is first called symbolically. CasADi sends the resulting graph through `pydot` to Graphviz; red boxes are symbolic inputs, green boxes are constants, and arrows show how dependencies feed each operation. The `direction="LR"` option lays the graph out from left to right. The helper writes `_temp.png` and `source.dot` in the working directory as intermediate files; both are ignored by Git.


`map` evaluates independent columns; `mapaccum` represents a recurrence. Both keep Python out of repeated evaluation and expose structure to CasADi. Function options such as `record_time`, `regularity_check`, and `post_expand` should be chosen deliberately and inspected with `print_options()` when needed.

**Common mistake:** a huge expanded graph can trade compilation and memory for a small evaluation speedup. Benchmark the actual workload. **Recap:** place reusable boundaries around models, stages, and residuals; map repeated structure; serialize only between compatible CasADi versions.
